**Momentum Breakout (2023-2026)**

In [ ]:
# 1. التسطيب السليم (بدون تعارض)
!pip install --upgrade pip -q
!pip install yfinance pandas==2.2.3 -q
!pip install pandas_ta --no-deps -q

import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np

# ==========================================
# ⚙️ إعدادات المحفظة والسوق الأمريكي
# ==========================================
TICKERS = ['NVDA', 'MSFT', 'AAPL', 'META']
TOTAL_CAPITAL = 100000.0
CAPITAL_PER_STOCK = TOTAL_CAPITAL / len(TICKERS)  # 25,000 دولار لكل سهم

SLIPPAGE_RATE = 0.0005  # 0.05% انزلاق سعري
COMMISSION_RATE = 0.0   # تداول مجاني

trade_log = []
portfolio_final_value = 0

print("🚀 بدء تشغيل محرك اختراق الزخم (US Momentum Breakout Engine)...")
print("📅 الفترة: 2023-01-01 إلى اليوم\n")

for ticker in TICKERS:
    print(f"🛠️ جاري سحب وتحليل بيانات {ticker}...")

    # سحب الداتا
    df = yf.download(ticker, start="2023-01-01", progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df = df.xs(ticker, axis=1, level=1)

    # المؤشرات
    df['20D_High'] = df['High'].rolling(window=20).max().shift(1)
    df['Vol_SMA_20'] = ta.sma(df['Volume'], length=20)
    df['SMA_50'] = ta.sma(df['Close'], length=50)
    df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    df.dropna(inplace=True)

    # محرك التداول
    cash = CAPITAL_PER_STOCK
    shares = 0
    highest_price_since_buy = 0

    for index, row in df.iterrows():
        current_price = row['Close']

        # --- شروط الدخول ---
        if shares == 0:
            breakout = current_price > row['20D_High']
            high_volume = row['Volume'] > (1.5 * row['Vol_SMA_20'])
            uptrend = current_price > row['SMA_50']

            if breakout and high_volume and uptrend:
                execution_price = current_price * (1 + SLIPPAGE_RATE)
                shares_to_buy = int(cash / execution_price)

                if shares_to_buy > 0:
                    trade_value = shares_to_buy * execution_price
                    cash -= trade_value
                    shares += shares_to_buy
                    highest_price_since_buy = execution_price

                    trade_log.append({
                        "Ticker": ticker,
                        "Date": index.strftime('%Y-%m-%d'),
                        "Action": "🟢 BUY",
                        "Price_USD": round(execution_price, 2),
                        "Shares": shares_to_buy,
                        "Trade_Value_USD": round(trade_value, 2),
                        "Remaining_Cash_USD": round(cash, 2),
                        "Reason": "Volume Breakout"
                    })

        # --- شروط الخروج ---
        elif shares > 0:
            if current_price > highest_price_since_buy:
                highest_price_since_buy = current_price

            trailing_stop = highest_price_since_buy - (2.5 * row['ATR'])

            if current_price < trailing_stop:
                execution_price = current_price * (1 - SLIPPAGE_RATE)
                trade_value = shares * execution_price
                cash += trade_value

                trade_log.append({
                    "Ticker": ticker,
                    "Date": index.strftime('%Y-%m-%d'),
                    "Action": "🔴 SELL",
                    "Price_USD": round(execution_price, 2),
                    "Shares": shares,
                    "Trade_Value_USD": round(trade_value, 2),
                    "Remaining_Cash_USD": round(cash, 2),
                    "Reason": "Trailing Stop Hit"
                })
                shares = 0
                highest_price_since_buy = 0

    # تقييم السهم في نهاية الفترة
    final_stock_value = cash + (shares * df.iloc[-1]['Close'])
    portfolio_final_value += final_stock_value
    stock_roi = ((final_stock_value - CAPITAL_PER_STOCK) / CAPITAL_PER_STOCK) * 100
    print(f"✅ {ticker}: {final_stock_value:,.2f} USD (عائد {stock_roi:.2f}%)")

# ==========================================
# 📈 التقرير النهائي
# ==========================================
net_profit = portfolio_final_value - TOTAL_CAPITAL
total_roi = (net_profit / TOTAL_CAPITAL) * 100

print("="*50)
print(f"📊 النتيجة النهائية لمحفظة (Mag 7):")
print(f"صافي الربح: {net_profit:,.2f} USD ({total_roi:.2f}%)")
print("="*50)

if trade_log:
    df_trades = pd.DataFrame(trade_log)
    df_trades.to_excel("US_Detailed_Momentum_Log.xlsx", index=False)
    print("✅ تم استخراج دفتر الصفقات التفصيلي في: US_Detailed_Momentum_Log.xlsx")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.9 MB/s eta 0:00:00
🚀 بدء تشغيل محرك اختراق الزخم (US Momentum Breakout Engine)...
📅 الفترة: 2023-01-01 إلى اليوم

🛠️ جاري سحب وتحليل بيانات NVDA...


/tmp/ipykernel_4038/2245663546.py:31: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2023-01-01", progress=False)


✅ NVDA: 72,587.52 USD (عائد 190.35%)
🛠️ جاري سحب وتحليل بيانات MSFT...


/tmp/ipykernel_4038/2245663546.py:31: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2023-01-01", progress=False)


✅ MSFT: 32,609.89 USD (عائد 30.44%)
🛠️ جاري سحب وتحليل بيانات AAPL...


/tmp/ipykernel_4038/2245663546.py:31: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2023-01-01", progress=False)


✅ AAPL: 30,602.44 USD (عائد 22.41%)
🛠️ جاري سحب وتحليل بيانات META...


/tmp/ipykernel_4038/2245663546.py:31: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2023-01-01", progress=False)


✅ META: 23,597.93 USD (عائد -5.61%)
📊 النتيجة النهائية لمحفظة (Mag 7):
صافي الربح: 59,397.77 USD (59.40%)
✅ تم استخراج دفتر الصفقات التفصيلي في: US_Detailed_Momentum_Log.xlsx


**Market Context Agent-Report**

In [ ]:
!pip install groq duckduckgo-search pandas openpyxl -q

import pandas as pd
from groq import Groq
from duckduckgo_search import DDGS
import time

# ==========================================
# ⚙️ الإعدادات
# ==========================================
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"
client = Groq(api_key=GROQ_API_KEY)

try:
    # 1. قراءة شيت الصفقات اللي طلعناه من الكود اللي فات
    df = pd.read_excel("US_Detailed_Momentum_Log.xlsx")

    # 2. فلترة صفقات الشراء بتاعة NVDA
    nvda_buys = df[(df['Ticker'] == 'NVDA') & (df['Action'] == '🟢 BUY')].copy()

    # هناخد أول 4 موجات بس عشان نحافظ على كوتا الـ API
    nvda_buys = nvda_buys.head(4)

    report_data = []

    print("🕵️ جاري تشغيل وكيل الأخبار (Market Context Agent)...\n")
    print("⚠️ الكود هيشتغل ببطء (4 ثواني بين كل موجة) لحماية الكوتا...\n")

    for index, row in nvda_buys.iterrows():
        trade_date = row['Date']
        buy_price = row['Price_USD']

        print(f"🔍 ببحث عن أحداث إنفيديا يوم: {trade_date}...")

        # 3. سحب الأخبار من محرك البحث في التاريخ ده
        search_query = f"Nvidia NVDA stock news {trade_date}"
        headlines = []
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(search_query, max_results=3))
                headlines = [res['title'] for res in results]
        except Exception as e:
            headlines = ["No specific news found"]

        # 4. توجيه الذكاء الاصطناعي للتحليل (The Prompt)
        prompt = f"""
        أنت محلل مالي خبير في وول ستريت.
        يوم {trade_date}، خوارزمية التداول الخاصة بنا اكتشفت انفجار سعري (Volume Breakout) في سهم إنفيديا (NVDA) واشترت السهم بسعر {buy_price}$.
        عناوين الأخبار في نفس الوقت كانت: {headlines}

        لخص في "سطرين فقط باللغة العربية" الحدث الأساسي أو المبرر الاقتصادي/التكنولوجي الذي دفع المؤسسات لشراء السهم بقوة في هذا التاريخ.
        """

        try:
            chat_completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="llama3-8b-8192",
                temperature=0.3, # درجة إبداع قليلة عشان الرد يكون دقيق
            )
            analysis = chat_completion.choices[0].message.content.strip()
        except Exception as e:
            analysis = "حدث خطأ في استدعاء الذكاء الاصطناعي بسبب الكوتا."

        report_data.append({
            "Date": trade_date,
            "Entry_Price_USD": buy_price,
            "News_Headlines": " | ".join(headlines),
            "Agent_Context_Analysis": analysis
        })

        # فرامل لحماية الكوتا
        time.sleep(4)

    # 5. حفظ التقرير في شيت إكسيل جديد
    report_df = pd.DataFrame(report_data)
    report_df.to_excel("NVDA_Waves_Context_Report.xlsx", index=False)

    print("\n" + "="*50)
    print("✅ التقرير الإخباري جاهز! إليك ملخص الموجات:")
    print("="*50)

    for item in report_data:
        print(f"📅 موجة {item['Date']} (سعر الدخول: {item['Entry_Price_USD']}$):")
        print(f"💡 التفسير: {item['Agent_Context_Analysis']}\n")
        print("-" * 30)

except FileNotFoundError:
    print("❌ ملف US_Detailed_Momentum_Log.xlsx مش موجود. رن الكود اللي فات الأول عشان يطلعه.")

🕵️ جاري تشغيل وكيل الأخبار (Market Context Agent)...

⚠️ الكود هيشتغل ببطء (4 ثواني بين كل موجة) لحماية الكوتا...

🔍 ببحث عن أحداث إنفيديا يوم: 2023-03-17...


/tmp/ipykernel_4038/4225320080.py:39: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use tim

🔍 ببحث عن أحداث إنفيديا يوم: 2024-01-08...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 ببحث عن أحداث إنفيديا يوم: 2024-02-22...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 ببحث عن أحداث إنفيديا يوم: 2024-05-23...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


✅ التقرير الإخباري جاهز! إليك ملخص الموجات:
📅 موجة 2023-03-17 (سعر الدخول: 25.66$):
💡 التفسير: حدث خطأ في استدعاء الذكاء الاصطناعي بسبب الكوتا.

------------------------------
📅 موجة 2024-01-08 (سعر الدخول: 52.13$):
💡 التفسير: حدث خطأ في استدعاء الذكاء الاصطناعي بسبب الكوتا.

------------------------------
📅 موجة 2024-02-22 (سعر الدخول: 78.35$):
💡 التفسير: حدث خطأ في استدعاء الذكاء الاصطناعي بسبب الكوتا.

------------------------------
📅 موجة 2024-05-23 (سعر الدخول: 103.56$):
💡 التفسير: حدث خطأ في استدعاء الذكاء الاصطناعي بسبب الكوتا.

------------------------------


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Bear Market Stress Test - 2022**
اربط الحزام يا هندسة، إحنا داخلين على أظلم سنة في تاريخ التكنولوجيا الحديث (2022).

في السنة دي، الفيدرالي الأمريكي رفع الفايدة بجنون، ومؤشر الناسداك (الخاص بالتكنولوجيا) خسر أكتر من 33% من قيمته، وسهم زي ميتا (META) نزل أكتر من 64%! أي حد كان شغال بنظام (اشتري واحتفظ - Buy and Hold) محفظته اتفرمت.

الهدف من "اختبار الجحيم" ده مش إننا نعمل أرباح خرافية (لأن التريند أصلاً هابط)، الهدف الأساسي هو (حماية رأس المال - Capital Preservation). عايزين نشوف هل الـ (SMA 50) و(وقف الخسارة المتحرك) هيمنعوا البوت من الدخول في فخ الهبوط ولا لأ؟


In [ ]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np

# ==========================================
# ⚙️ إعدادات المحفظة
# ==========================================
TICKERS = ['NVDA', 'MSFT', 'AAPL', 'META']
TOTAL_CAPITAL = 100000.0
CAPITAL_PER_STOCK = TOTAL_CAPITAL / len(TICKERS)  # 25 ألف لكل سهم

SLIPPAGE_RATE = 0.0005
COMMISSION_RATE = 0.0

trade_log = []
portfolio_final_value = 0

print("🔥 بدء اختبار الجحيم: محرك الزخم في السوق الهابط (Bear Market - 2022)...\n")

for ticker in TICKERS:
    print(f"🛠️ جاري سحب وتحليل بيانات {ticker} لعام 2022...")

    # خد بالك: بنسحب الداتا من أواخر 2021 عشان المؤشرات (زي SMA 50) تلحق تحسب وتكون جاهزة من أول يوم في 2022
    df = yf.download(ticker, start="2021-10-01", end="2022-12-31", progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df = df.xs(ticker, axis=1, level=1)

    # المؤشرات
    df['20D_High'] = df['High'].rolling(window=20).max().shift(1)
    df['Vol_SMA_20'] = ta.sma(df['Volume'], length=20)
    df['SMA_50'] = ta.sma(df['Close'], length=50)
    df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    df.dropna(inplace=True)

    # فلترة الداتا عشان التداول يبدأ فعلياً من 1 يناير 2022
    df = df[df.index >= '2022-01-01']

    cash = CAPITAL_PER_STOCK
    shares = 0
    highest_price_since_buy = 0

    for index, row in df.iterrows():
        current_price = row['Close']

        # --- شروط الدخول ---
        if shares == 0:
            breakout = current_price > row['20D_High']
            high_volume = row['Volume'] > (1.5 * row['Vol_SMA_20'])
            uptrend = current_price > row['SMA_50']

            if breakout and high_volume and uptrend:
                execution_price = current_price * (1 + SLIPPAGE_RATE)
                shares_to_buy = int(cash / execution_price)

                if shares_to_buy > 0:
                    trade_value = shares_to_buy * execution_price
                    cash -= trade_value
                    shares += shares_to_buy
                    highest_price_since_buy = execution_price

                    trade_log.append({
                        "Ticker": ticker,
                        "Date": index.strftime('%Y-%m-%d'),
                        "Action": "🟢 BUY",
                        "Price_USD": round(execution_price, 2),
                        "Shares": shares_to_buy,
                        "Trade_Value_USD": round(trade_value, 2),
                        "Remaining_Cash_USD": round(cash, 2),
                        "Reason": "Volume Breakout"
                    })

        # --- شروط الخروج ---
        elif shares > 0:
            if current_price > highest_price_since_buy:
                highest_price_since_buy = current_price

            trailing_stop = highest_price_since_buy - (2.5 * row['ATR'])

            if current_price < trailing_stop:
                execution_price = current_price * (1 - SLIPPAGE_RATE)
                trade_value = shares * execution_price
                cash += trade_value

                trade_log.append({
                    "Ticker": ticker,
                    "Date": index.strftime('%Y-%m-%d'),
                    "Action": "🔴 SELL",
                    "Price_USD": round(execution_price, 2),
                    "Shares": shares,
                    "Trade_Value_USD": round(trade_value, 2),
                    "Remaining_Cash_USD": round(cash, 2),
                    "Reason": "Trailing Stop Hit"
                })
                shares = 0
                highest_price_since_buy = 0

    # تقييم السهم في نهاية 2022
    # لو فضل أسهم معانا هنحسب قيمتها، لو مفيش يبقى الكاش المتبقي
    final_stock_value = cash + (shares * df.iloc[-1]['Close']) if not df.empty else cash
    portfolio_final_value += final_stock_value
    stock_roi = ((final_stock_value - CAPITAL_PER_STOCK) / CAPITAL_PER_STOCK) * 100
    print(f"✅ {ticker}: {final_stock_value:,.2f} USD (عائد {stock_roi:.2f}%)")

# ==========================================
# 📈 التقرير النهائي
# ==========================================
net_profit = portfolio_final_value - TOTAL_CAPITAL
total_roi = (net_profit / TOTAL_CAPITAL) * 100

print("="*50)
print(f"📊 نتيجة محفظة (Mag 7) في 2022 (سنة الانهيار):")
print(f"صافي الربح/الخسارة: {net_profit:,.2f} USD ({total_roi:.2f}%)")
print(f"إجمالي العمليات: {len(trade_log)}")
print("="*50)

# حفظ ملف الإكسيل المزدوج (شيت الصفقات + شيت الملخص)
if trade_log:
    with pd.ExcelWriter("US_Bear_Market_2022.xlsx") as writer:
        # الشيت الأول: التفاصيل
        df_trades = pd.DataFrame(trade_log)
        df_trades.to_excel(writer, sheet_name="Trade_Log", index=False)

        # الشيت التاني: الملخص الإداري
        summary_data = {
            "Metric": ["Initial Capital", "Final Value", "Net Profit/Loss", "ROI (%)", "Total Trades"],
            "Value": [TOTAL_CAPITAL, round(portfolio_final_value, 2), round(net_profit, 2), round(total_roi, 2), len(trade_log)]
        }
        pd.DataFrame(summary_data).to_excel(writer, sheet_name="Executive_Summary", index=False)

    print("✅ تم استخراج التقرير الشامل في: US_Bear_Market_2022.xlsx")
else:
    print("⚠️ لم يقم المحرك بأي عمليات تداول (البوت رفض يدخل في السوق الهابط!).")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔥 بدء اختبار الجحيم: محرك الزخم في السوق الهابط (Bear Market - 2022)...

🛠️ جاري سحب وتحليل بيانات NVDA لعام 2022...
✅ NVDA: 21,655.32 USD (عائد -13.38%)
🛠️ جاري سحب وتحليل بيانات MSFT لعام 2022...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/tmp/ipykernel_4038/2137587502.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2021-10-01", end="2022-12-31", progress=False)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in 

✅ MSFT: 23,331.08 USD (عائد -6.68%)
🛠️ جاري سحب وتحليل بيانات AAPL لعام 2022...
✅ AAPL: 22,280.93 USD (عائد -10.88%)
🛠️ جاري سحب وتحليل بيانات META لعام 2022...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

✅ META: 25,000.00 USD (عائد 0.00%)
📊 نتيجة محفظة (Mag 7) في 2022 (سنة الانهيار):
صافي الربح/الخسارة: -7,732.67 USD (-7.73%)
إجمالي العمليات: 6
✅ تم استخراج التقرير الشامل في: US_Bear_Market_2022.xlsx


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/openpyxl/cell/cell.py:164: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x79fb715b35b0>
  if next(ILLEGAL_CHARACTERS_RE.finditer(value), None):
/usr/local/lib/python3.13/dist-packages/openpyxl/cell/cell.py:164: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x79fb7112c5e0>
  if next(ILLEGAL_CHARACTERS_RE.finditer(value), None):
/usr/local/lib/python3.13/dist-packages/openpyxl/cell/cell.py:164: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x79fb7112c9a0>
  if next(ILLEGAL_CHARACTERS_RE.finditer(value), None):
/usr/local/lib/python3.13/dist-packages/openpyxl/cell/cell

**Statistical Arbitrage Engine-Pairs Trading**

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# ==========================================
# ⚙️ إعدادات المحفظة والتوأم (The Pair)
# ==========================================
TICKER_A = 'AAPL'
TICKER_B = 'MSFT'
INITIAL_CAPITAL = 100000.0

print(f"🚀 بدء تشغيل محرك التداول المزدوج (Pairs Trading) على {TICKER_A} و {TICKER_B}...")
print("📅 الفترة: 2022-01-01 إلى اليوم (اختبار شامل للانهيار والزخم معاً)\n")

# 1. سحب بيانات السهمين
df_a = yf.download(TICKER_A, start="2022-01-01", progress=False)['Close']
df_b = yf.download(TICKER_B, start="2022-01-01", progress=False)['Close']

if isinstance(df_a, pd.DataFrame): df_a = df_a.squeeze()
if isinstance(df_b, pd.DataFrame): df_b = df_b.squeeze()

df = pd.DataFrame({TICKER_A: df_a, TICKER_B: df_b}).dropna()

# 2. حساب الفروقات الإحصائية (Z-Score)
# بنقسم سعر أبل على مايكروسوفت عشان نجيب (النسبة)
df['Ratio'] = df[TICKER_A] / df[TICKER_B]

# بنحسب متوسط النسبة في آخر 30 يوم، والانحراف المعياري
df['Mean'] = df['Ratio'].rolling(window=30).mean()
df['Std'] = df['Ratio'].rolling(window=30).std()

# الـ Z-Score بيقولنا: النسبة دي بعيدة قد إيه عن الطبيعي بتاعها؟
df['Z_Score'] = (df['Ratio'] - df['Mean']) / df['Std']

# 3. محرك التداول
capital = INITIAL_CAPITAL
position = 0  # 0: مفيش صفقة | 1: شاري أبل وبايع مايكروسوفت | -1: بايع أبل وشاري مايكروسوفت
trade_log = []
entry_price_a = 0
entry_price_b = 0

for index, row in df.iterrows():
    z = row['Z_Score']
    if pd.isna(z): continue

    # --- شروط الدخول ---
    if position == 0:
        if z < -2.0:
            # أبل رخيص جداً مقارنة بمايكروسوفت
            position = 1
            entry_price_a = row[TICKER_A]
            entry_price_b = row[TICKER_B]
            trade_log.append({
                "Date": index.strftime('%Y-%m-%d'),
                "Action": f"🟢 LONG {TICKER_A} / 🔴 SHORT {TICKER_B}",
                "Z_Score": round(z, 2),
                "PnL_USD": 0.0,
                "Capital_USD": round(capital, 2)
            })

        elif z > 2.0:
            # أبل غالي جداً مقارنة بمايكروسوفت
            position = -1
            entry_price_a = row[TICKER_A]
            entry_price_b = row[TICKER_B]
            trade_log.append({
                "Date": index.strftime('%Y-%m-%d'),
                "Action": f"🔴 SHORT {TICKER_A} / 🟢 LONG {TICKER_B}",
                "Z_Score": round(z, 2),
                "PnL_USD": 0.0,
                "Capital_USD": round(capital, 2)
            })

    # --- شروط الخروج (العودة للمتوسط Z = 0) ---
    elif position == 1 and z >= 0:
        # نقفل الصفقة ونحسب الأرباح
        ret_a = (row[TICKER_A] - entry_price_a) / entry_price_a # مكسب الشراء
        ret_b = (entry_price_b - row[TICKER_B]) / entry_price_b # مكسب البيع على المكشوف

        trade_pnl = (capital / 2) * ret_a + (capital / 2) * ret_b
        capital += trade_pnl
        position = 0

        trade_log.append({
            "Date": index.strftime('%Y-%m-%d'),
            "Action": "⚪ CLOSE POSITIONS",
            "Z_Score": round(z, 2),
            "PnL_USD": round(trade_pnl, 2),
            "Capital_USD": round(capital, 2)
        })

    elif position == -1 and z <= 0:
        # نقفل الصفقة ونحسب الأرباح
        ret_a = (entry_price_a - row[TICKER_A]) / entry_price_a # مكسب البيع على المكشوف
        ret_b = (row[TICKER_B] - entry_price_b) / entry_price_b # مكسب الشراء

        trade_pnl = (capital / 2) * ret_a + (capital / 2) * ret_b
        capital += trade_pnl
        position = 0

        trade_log.append({
            "Date": index.strftime('%Y-%m-%d'),
            "Action": "⚪ CLOSE POSITIONS",
            "Z_Score": round(z, 2),
            "PnL_USD": round(trade_pnl, 2),
            "Capital_USD": round(capital, 2)
        })

# ==========================================
# 📈 التقرير النهائي
# ==========================================
net_profit = capital - INITIAL_CAPITAL
total_roi = (net_profit / INITIAL_CAPITAL) * 100
closed_trades = len([t for t in trade_log if t['Action'] == '⚪ CLOSE POSITIONS'])

print("="*50)
print(f"📊 نتيجة التداول المزدوج (Market Neutral) 2022-2026:")
print(f"صافي الربح: {net_profit:,.2f} USD ({total_roi:.2f}%)")
print(f"إجمالي الصفقات المغلقة: {closed_trades}")
print("="*50)

if trade_log:
    with pd.ExcelWriter("US_Pairs_Trading_Arbitrage.xlsx") as writer:
        df_trades = pd.DataFrame(trade_log)
        df_trades.to_excel(writer, sheet_name="Trade_Log", index=False)

        summary_data = {
            "Metric": ["Strategy", "Pair", "Initial Capital", "Final Value", "Net Profit", "ROI (%)", "Closed Trades"],
            "Value": ["Statistical Arbitrage (Market Neutral)", f"{TICKER_A} / {TICKER_B}", INITIAL_CAPITAL, round(capital, 2), round(net_profit, 2), round(total_roi, 2), closed_trades]
        }
        pd.DataFrame(summary_data).to_excel(writer, sheet_name="Executive_Summary", index=False)

    print("✅ تم استخراج السجل الشامل في: US_Pairs_Trading_Arbitrage.xlsx")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🚀 بدء تشغيل محرك التداول المزدوج (Pairs Trading) على AAPL و MSFT...
📅 الفترة: 2022-01-01 إلى اليوم (اختبار شامل للانهيار والزخم معاً)



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

📊 نتيجة التداول المزدوج (Market Neutral) 2022-2026:
صافي الربح: 20,698.28 USD (20.70%)
إجمالي الصفقات المغلقة: 31
✅ تم استخراج السجل الشامل في: US_Pairs_Trading_Arbitrage.xlsx


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Pairs Trading-Srop-Loss**

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# ==========================================
# ⚙️ إعدادات المحفظة والتوأم
# ==========================================
TICKER_A = 'AAPL'
TICKER_B = 'MSFT'
INITIAL_CAPITAL = 100000.0

# محددات الإحصاء
ENTRY_Z = 2.0
EXIT_Z = 0.0
STOP_LOSS_Z = 4.0  # 🚨 فرامل الطوارئ (لو الانحراف وسع جداً)

print(f"🚀 تشغيل محرك التداول المزدوج V2 (مُحصن بالـ Stop Loss)...")

# 1. سحب البيانات
df_a = yf.download(TICKER_A, start="2022-01-01", progress=False)['Close']
df_b = yf.download(TICKER_B, start="2022-01-01", progress=False)['Close']

if isinstance(df_a, pd.DataFrame): df_a = df_a.squeeze()
if isinstance(df_b, pd.DataFrame): df_b = df_b.squeeze()

df = pd.DataFrame({TICKER_A: df_a, TICKER_B: df_b}).dropna()

# 2. حساب المؤشرات الإحصائية
df['Ratio'] = df[TICKER_A] / df[TICKER_B]
df['Mean'] = df['Ratio'].rolling(window=30).mean()
df['Std'] = df['Ratio'].rolling(window=30).std()
df['Z_Score'] = (df['Ratio'] - df['Mean']) / df['Std']

# 3. محرك التداول
capital = INITIAL_CAPITAL
position = 0
trade_log = []
entry_price_a = 0
entry_price_b = 0

for index, row in df.iterrows():
    z = row['Z_Score']
    if pd.isna(z): continue

    # --- شروط الدخول ---
    if position == 0:
        if z < -ENTRY_Z:
            position = 1
            entry_price_a = row[TICKER_A]
            entry_price_b = row[TICKER_B]
            trade_log.append({
                "Date": index.strftime('%Y-%m-%d'),
                "Action": f"🟢 LONG {TICKER_A} / 🔴 SHORT {TICKER_B}",
                "Z_Score": round(z, 2),
                "PnL_USD": 0.0,
                "Capital_USD": round(capital, 2),
                "Reason": "Entry Z < -2"
            })

        elif z > ENTRY_Z:
            position = -1
            entry_price_a = row[TICKER_A]
            entry_price_b = row[TICKER_B]
            trade_log.append({
                "Date": index.strftime('%Y-%m-%d'),
                "Action": f"🔴 SHORT {TICKER_A} / 🟢 LONG {TICKER_B}",
                "Z_Score": round(z, 2),
                "PnL_USD": 0.0,
                "Capital_USD": round(capital, 2),
                "Reason": "Entry Z > 2"
            })

    # --- شروط الخروج ---
    elif position == 1:
        # خروج بربح (العودة للمتوسط) أو خروج بخسارة (كسر الفرامل)
        if z >= EXIT_Z or z <= -STOP_LOSS_Z:
            ret_a = (row[TICKER_A] - entry_price_a) / entry_price_a
            ret_b = (entry_price_b - row[TICKER_B]) / entry_price_b
            trade_pnl = (capital / 2) * ret_a + (capital / 2) * ret_b
            capital += trade_pnl

            reason = "Take Profit (Mean Reversion)" if z >= EXIT_Z else "🚨 STOP LOSS (Structural Break)"

            trade_log.append({
                "Date": index.strftime('%Y-%m-%d'),
                "Action": "⚪ CLOSE POSITIONS",
                "Z_Score": round(z, 2),
                "PnL_USD": round(trade_pnl, 2),
                "Capital_USD": round(capital, 2),
                "Reason": reason
            })
            position = 0

    elif position == -1:
        # خروج بربح أو طوارئ
        if z <= EXIT_Z or z >= STOP_LOSS_Z:
            ret_a = (entry_price_a - row[TICKER_A]) / entry_price_a
            ret_b = (row[TICKER_B] - entry_price_b) / entry_price_b
            trade_pnl = (capital / 2) * ret_a + (capital / 2) * ret_b
            capital += trade_pnl

            reason = "Take Profit (Mean Reversion)" if z <= EXIT_Z else "🚨 STOP LOSS (Structural Break)"

            trade_log.append({
                "Date": index.strftime('%Y-%m-%d'),
                "Action": "⚪ CLOSE POSITIONS",
                "Z_Score": round(z, 2),
                "PnL_USD": round(trade_pnl, 2),
                "Capital_USD": round(capital, 2),
                "Reason": reason
            })
            position = 0

# ==========================================
# 📈 التقرير النهائي
# ==========================================
net_profit = capital - INITIAL_CAPITAL
total_roi = (net_profit / INITIAL_CAPITAL) * 100
closed_trades = [t for t in trade_log if t['Action'] == '⚪ CLOSE POSITIONS']

print("="*50)
print(f"📊 نتيجة التداول المزدوج V2 (بعد سد الثغرة):")
print(f"صافي الربح: {net_profit:,.2f} USD ({total_roi:.2f}%)")
print(f"إجمالي الصفقات المغلقة: {len(closed_trades)}")
if closed_trades:
    max_loss = min([t['PnL_USD'] for t in closed_trades])
    print(f"أقصى خسارة في صفقة واحدة: {max_loss:,.2f} USD")
print("="*50)

if trade_log:
    with pd.ExcelWriter("US_Pairs_Trading_V2_Fixed.xlsx") as writer:
        pd.DataFrame(trade_log).to_excel(writer, sheet_name="Trade_Log", index=False)
    print("✅ تم حفظ السجل المُحصن في: US_Pairs_Trading_V2_Fixed.xlsx")

🚀 تشغيل محرك التداول المزدوج V2 (مُحصن بالـ Stop Loss)...


/tmp/ipykernel_3978/2794166514.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_a = yf.download(TICKER_A, start="2022-01-01", progress=False)['Close']
/tmp/ipykernel_3978/2794166514.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_b = yf.download(TICKER_B, start="2022-01-01", progress=False)['Close']


📊 نتيجة التداول المزدوج V2 (بعد سد الثغرة):
صافي الربح: 20,698.36 USD (20.70%)
إجمالي الصفقات المغلقة: 31
أقصى خسارة في صفقة واحدة: -15,381.92 USD
✅ تم حفظ السجل المُحصن في: US_Pairs_Trading_V2_Fixed.xlsx
